In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:,.2f}'.format)

sns.set_theme(style='whitegrid')
BLUE, GREY = '#2F6DB5', '#9AA5B1'          # one main colour + one neutral
plt.rcParams['figure.figsize'] = (9, 4.5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'

import os
os.makedirs('images', exist_ok=True)


In [ ]:
# encoding='utf-8-sig' removes an invisible character (BOM) that would otherwise
# stick to the first column name, e.g. 'CustomerID'
customers   = pd.read_csv('customers.csv',   encoding='utf-8-sig')
calendar    = pd.read_csv('datatable.csv',   encoding='utf-8-sig')
orders      = pd.read_csv('orders.csv',      encoding='utf-8-sig')
restaurants = pd.read_csv('restaurants.csv', encoding='utf-8-sig', low_memory=False)

tables = {'customers': customers, 'calendar': calendar,
          'orders': orders, 'restaurants': restaurants}

for name, df in tables.items():
    print(f'{name:12s} rows = {df.shape[0]:>7,}   columns = {df.shape[1]}')


In [ ]:
for name, df in tables.items():
    print('=' * 70)
    print(name.upper(), df.shape)
    display(df.head(3))
    df.info()


In [ ]:
customers['SignupDate'] = pd.to_datetime(customers['SignupDate'])
calendar['Date']        = pd.to_datetime(calendar['Date'])
orders['OrderDate']     = pd.to_datetime(orders['OrderDate'])

print('Customers signed up :', customers['SignupDate'].min().date(), '->', customers['SignupDate'].max().date())
print('Orders placed       :', orders['OrderDate'].min().date(),     '->', orders['OrderDate'].max().date())
print('Calendar table      :', calendar['Date'].min().date(),        '->', calendar['Date'].max().date())



In [ ]:
missing = pd.DataFrame({
    'missing_count': restaurants.isna().sum(),
    'missing_pct':   restaurants.isna().mean() * 100
}).sort_values('missing_pct', ascending=False)

print('Total missing values in the other three tables:',
      customers.isna().sum().sum(), calendar.isna().sum().sum(), orders.isna().sum().sum())
display(missing[missing['missing_count'] > 0])

ax = missing[missing['missing_pct'] > 0]['missing_pct'].sort_values().plot(kind='barh', color=BLUE)
ax.set_title('Missing values in the raw restaurants table')
ax.set_xlabel('% of rows missing')
plt.tight_layout()
plt.savefig('images/06_missing_values.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
for name, df in tables.items():
    print(f'{name:12s} fully duplicated rows: {df.duplicated().sum()}')

print()
print('CustomerID unique   :', customers['CustomerID'].is_unique)
print('OrderID unique      :', orders['OrderID'].is_unique)
print('RestaurantID unique :', restaurants['RestaurantID'].is_unique)



In [ ]:
# In a healthy row, online_order and book_table are only "Yes" or "No".
valid_mask = (restaurants['online_order'].isin(['Yes', 'No']) &
              restaurants['book_table'].isin(['Yes', 'No']))

n_bad = (~valid_mask).sum()
print(f'Healthy rows   : {valid_mask.sum():,}')
print(f'Corrupted rows : {n_bad:,}  ({n_bad / len(restaurants):.1%} of the table)')

print('\nWhat a corrupted row looks like (first 110 characters of online_order):')
print(repr(str(restaurants.loc[~valid_mask, 'online_order'].iloc[0])[:110]))


In [ ]:
rest = restaurants[valid_mask].copy()

# '4.1/5' and '4.1 /5' -> 4.1   ('NEW' and '-' become NaN because they are not numbers)
rest['rating'] = (rest['rate'].astype(str)
                  .str.replace('/5', '', regex=False)
                  .str.strip()
                  .pipe(pd.to_numeric, errors='coerce'))

rest['votes'] = pd.to_numeric(rest['votes'], errors='coerce')

# '1,200' -> 1200
rest['cost_for_two'] = pd.to_numeric(
    rest['approx_cost(for two people)'].astype(str).str.replace(',', '', regex=False),
    errors='coerce')

rest = rest.rename(columns={'listed_in(type)': 'listed_type',
                            'listed_in(city)': 'listed_city'})

print('Clean restaurants table:', rest.shape)
display(rest[['rating', 'votes', 'cost_for_two']].describe().T)
print('\nRestaurants with no rating yet (NEW / missing):', rest['rating'].isna().sum())


In [ ]:
# (a) Orders that point to a restaurant we just removed
bad_ids = set(restaurants.loc[~valid_mask, 'RestaurantID'])
orders['restaurant_ok'] = ~orders['RestaurantID'].isin(bad_ids)
print(f"Orders linked to a corrupted restaurant row: {(~orders['restaurant_ok']).mean():.1%}")

# (b) Orders placed BEFORE the customer signed up (logically impossible)
orders['signup_date']   = orders['CustomerID'].map(customers.set_index('CustomerID')['SignupDate'])
orders['before_signup'] = orders['OrderDate'] < orders['signup_date']
print(f"Orders placed before the customer's signup date: {orders['before_signup'].mean():.1%}")


In [ ]:
master = (orders
          .merge(customers[['CustomerID', 'City', 'CustomerType', 'SignupDate']],
                 on='CustomerID', how='left')
          .merge(calendar[['Date', 'Year', 'MonthNo', 'MonthName', 'Quarter']],
                 left_on='OrderDate', right_on='Date', how='left')
          .merge(rest[['RestaurantID', 'name', 'location', 'rest_type', 'cuisines', 'rating',
                       'votes', 'cost_for_two', 'online_order', 'book_table',
                       'listed_type', 'listed_city']],
                 on='RestaurantID', how='left'))

master['DayOfWeek']    = master['OrderDate'].dt.day_name()
master['is_delivered'] = master['OrderStatus'].eq('Delivered')

assert len(master) == len(orders), 'A join duplicated rows!'
print('Master table:', master.shape)
print(f"Rows with restaurant details: {master['name'].notna().mean():.1%}")


In [ ]:
num_cols = ['OrderValue', 'DeliveryTimeMins', 'Quantity']
display(orders[num_cols].describe().T)
print('\nSkewness (0 = symmetric):')
print(orders[num_cols].skew().round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, num_cols):
    sns.histplot(orders[col], bins=30, discrete=(col == 'Quantity'), color=BLUE, ax=ax)
    ax.axvline(orders[col].median(), color='black', ls='--', lw=1, label='median')
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()



In [ ]:
status = orders['OrderStatus'].value_counts()
print((status / len(orders) * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
status.plot(kind='bar', color=[BLUE, GREY, GREY], ax=axes[0], rot=0)
axes[0].set_title('Orders by status')
axes[0].set_ylabel('number of orders')
for i, v in enumerate(status):
    axes[0].text(i, v, f'{v / len(orders):.1%}', ha='center', va='bottom')

ctype = customers['CustomerType'].value_counts()
ctype.plot(kind='bar', color=BLUE, ax=axes[1], rot=0)
axes[1].set_title('Customers by type')
axes[1].set_ylabel('number of customers')
for i, v in enumerate(ctype):
    axes[1].text(i, v, f'{v / len(customers):.1%}', ha='center', va='bottom')
plt.tight_layout()
plt.savefig('images/01_order_status.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# The last month in the data (Sep 2025) is incomplete, so we leave it out of the trend.
monthly = (master.groupby(master['OrderDate'].dt.to_period('M'))
                 .agg(orders=('OrderID', 'count')))
monthly['delivered_revenue'] = (master[master['is_delivered']]
                                .groupby(master['OrderDate'].dt.to_period('M'))['OrderValue'].sum())
monthly = monthly.iloc[:-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(monthly.index.astype(str), monthly['orders'], color=BLUE, marker='o', ms=3)
axes[0].set_ylim(0, monthly['orders'].max() * 1.2)
axes[0].set_title('Orders per month')
axes[0].set_xticks(range(0, len(monthly), 3))
axes[0].set_xticklabels(monthly.index.astype(str)[::3], rotation=45)

dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = master['DayOfWeek'].value_counts().reindex(dow_order)
axes[1].bar(dow.index.str[:3], dow.values, color=BLUE)
axes[1].set_title('Orders by day of week')
plt.tight_layout()
plt.savefig('images/02_order_trends.png', dpi=200, bbox_inches='tight')
plt.show()

print('Orders per quarter:')
display(pd.crosstab(master['Year'], master['Quarter']))
print(f"\nBiggest month: {monthly['orders'].idxmax()} ({monthly['orders'].max():,} orders)")
print(f"Smallest month: {monthly['orders'].idxmin()} ({monthly['orders'].min():,} orders)")
print(f"Weekday spread: {dow.min():,} to {dow.max():,} orders")


In [ ]:
opc = orders.groupby('CustomerID').size()
print(f'Customers with at least one order: {len(opc):,} of {len(customers):,}')
display(opc.describe().to_frame('orders_per_customer').T)

by_type = (master.groupby('CustomerType')
                 .agg(customers=('CustomerID', 'nunique'),
                      orders=('OrderID', 'count'),
                      avg_order_value=('OrderValue', 'mean'),
                      avg_delivery_min=('DeliveryTimeMins', 'mean'),
                      delivered_pct=('is_delivered', lambda s: s.mean() * 100)))
by_type['orders_per_customer'] = by_type['orders'] / by_type['customers']
display(by_type)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
sns.histplot(opc, bins=25, color=BLUE, ax=axes[0])
axes[0].set_title('Orders per customer')

top_city = customers['City'].value_counts().head(10).sort_values()
top_city.plot(kind='barh', color=BLUE, ax=axes[1])
axes[1].set_title('Top 10 customer areas')

customers['SignupDate'].dt.to_period('M').value_counts().sort_index().plot(color=BLUE, ax=axes[2])
axes[2].set_title('Signups per month')
axes[2].set_xlabel('')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

sns.histplot(rest['rating'].dropna(), bins=30, color=BLUE, ax=axes[0, 0])
axes[0, 0].set_title('Restaurant rating')

sns.histplot(rest['cost_for_two'].dropna(), bins=40, color=BLUE, log_scale=True, ax=axes[0, 1])
axes[0, 1].set_title('Cost for two (log scale)')

sns.histplot(rest['votes'].dropna(), bins=40, color=BLUE, log_scale=True, ax=axes[0, 2])
axes[0, 2].set_title('Votes (log scale)')

for ax, col in zip(axes[1], ['online_order', 'book_table', 'listed_type']):
    rest[col].value_counts().plot(kind='bar', color=BLUE, ax=ax, rot=30)
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig('images/08_distributions.png', dpi=200, bbox_inches='tight')
plt.show()

print('Share of restaurants accepting online orders :', f"{(rest['online_order'] == 'Yes').mean():.1%}")
print('Share offering table booking                 :', f"{(rest['book_table'] == 'Yes').mean():.1%}")
print('Median cost for two                          :', rest['cost_for_two'].median())


In [ ]:
print('Spearman correlations (works for skewed data):')
display(rest[['rating', 'votes', 'cost_for_two']].corr(method='spearman'))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sample = rest.dropna(subset=['rating', 'votes']).sample(5000, random_state=1)
axes[0].scatter(sample['votes'], sample['rating'], s=6, alpha=0.3, color=BLUE)
axes[0].set_xscale('log')
axes[0].set_xlabel('votes (log scale)')
axes[0].set_ylabel('rating')
axes[0].set_title('Rating vs votes')

sns.boxplot(data=rest, x='online_order', y='rating', color=BLUE, ax=axes[1])
axes[1].set_title('Rating: online ordering yes/no')

rest['cost_band'] = pd.qcut(rest['cost_for_two'], 4, labels=['Q1 cheapest', 'Q2', 'Q3', 'Q4 priciest'])
sns.boxplot(data=rest, x='cost_band', y='rating', color=BLUE, ax=axes[2])
axes[2].set_title('Rating by price band')
plt.tight_layout()
plt.savefig('images/03_rating_relationships.png', dpi=200, bbox_inches='tight')
plt.show()

print(rest.groupby('online_order')['rating'].agg(['count', 'mean', 'median']))
print(rest.groupby('cost_band', observed=True)['rating'].agg(['count', 'mean', 'median']))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.boxplot(data=orders, x='OrderStatus', y='DeliveryTimeMins', color=BLUE, ax=axes[0])
axes[0].set_title('Delivery time by order status')

sns.boxplot(data=orders, x='Quantity', y='OrderValue', color=BLUE, ax=axes[1])
axes[1].set_title('Order value by quantity')

corr = orders[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', vmin=-1, vmax=1, ax=axes[2])
axes[2].set_title('Correlation matrix')
plt.tight_layout()
plt.savefig('images/07_correlation.png', dpi=200, bbox_inches='tight')
plt.show()

print(orders.groupby('OrderStatus')[['OrderValue', 'DeliveryTimeMins']].mean().round(2))


In [ ]:
overall_cancel = (master['OrderStatus'] == 'Cancelled').mean() * 100
overall_delay  = (master['OrderStatus'] == 'Delayed').mean() * 100
print(f'Overall cancel rate: {overall_cancel:.2f}%   Overall delay rate: {overall_delay:.2f}%\n')

def problem_rates(col):
    t = pd.crosstab(master[col], master['OrderStatus'], normalize='index') * 100
    return t[['Cancelled', 'Delayed']].round(2)

for col in ['CustomerType', 'Quantity', 'DayOfWeek', 'listed_type']:
    print(f'--- by {col} ---')
    display(problem_rates(col))

rates = problem_rates('listed_type')
ax = rates['Cancelled'].sort_values().plot(kind='barh', color=BLUE, figsize=(8, 4),
                                           label='cancellation rate')
ax.axvline(overall_cancel, color='black', ls='--', label='overall average')
ax.set_xlim(0, 15)                       # leave empty room on the right for the legend
ax.set_title('Cancellation rate by restaurant listing type')
ax.set_xlabel('% of orders cancelled')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('images/04_cancellation_rate.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
alpha = 0.05

# 1) Chi-square: is order status related to customer type?
ct = pd.crosstab(master['CustomerType'], master['OrderStatus'])
chi2, p1, dof, _ = stats.chi2_contingency(ct)
cramers_v = np.sqrt(chi2 / (ct.values.sum() * (min(ct.shape) - 1)))
print(f'1) Status vs customer type  : chi2={chi2:.2f}, p={p1:.4f}, Cramer V={cramers_v:.4f}')

# 2) ANOVA: does delivery time differ between statuses?
groups = [g['DeliveryTimeMins'].values for _, g in orders.groupby('OrderStatus')]
f, p2 = stats.f_oneway(*groups)
print(f'2) Delivery time by status  : F={f:.2f}, p={p2:.4f}')

# 3) Kruskal-Wallis: does order value differ between customer types?
groups = [g['OrderValue'].values for _, g in orders.merge(customers[['CustomerID', 'CustomerType']]).groupby('CustomerType')]
h, p3 = stats.kruskal(*groups)
print(f'3) Order value by cust. type: H={h:.2f}, p={p3:.4f}')

# 4) Pearson: order value vs quantity
r, p4 = stats.pearsonr(orders['OrderValue'], orders['Quantity'])
print(f'4) Value vs quantity        : r={r:.4f}, p={p4:.4f}')

# 5) Mann-Whitney: do online-ordering restaurants have different ratings?
a = rest.loc[rest['online_order'] == 'Yes', 'rating'].dropna()
b = rest.loc[rest['online_order'] == 'No',  'rating'].dropna()
u, p5 = stats.mannwhitneyu(a, b)
print(f'5) Rating online yes vs no  : p={p5:.2e}, median yes={a.median()}, median no={b.median()}')

print('\nVerdicts (alpha = 0.05):')
for label, p in [('Status ~ customer type', p1), ('Delivery time ~ status', p2),
                 ('Order value ~ customer type', p3), ('Order value ~ quantity', p4),
                 ('Rating ~ online ordering', p5)]:
    print(f"  {label:30s}", 'SIGNIFICANT' if p < alpha else 'no evidence of a difference')


In [ ]:
def iqr_outliers(s):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    low, high = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
    n = ((s < low) | (s > high)).sum()
    return pd.Series({'lower_fence': low, 'upper_fence': high,
                      'outliers': n, 'outlier_pct': n / len(s) * 100, 'max': s.max()})

out = pd.DataFrame({
    'OrderValue':       iqr_outliers(orders['OrderValue']),
    'DeliveryTimeMins': iqr_outliers(orders['DeliveryTimeMins']),
    'cost_for_two':     iqr_outliers(rest['cost_for_two']),
    'votes':            iqr_outliers(rest['votes']),
}).T
display(out)

print('\nMost expensive restaurants (cost for two):')
display(rest.nlargest(5, 'cost_for_two')[['name', 'location', 'rest_type', 'cost_for_two', 'rating']])


In [ ]:
deliv = master[master['is_delivered']]

rev_type = deliv.groupby('listed_type')['OrderValue'].sum().sort_values()
top_loc  = deliv.groupby('location')['OrderValue'].sum().nlargest(10).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
(rev_type / 1e6).plot(kind='barh', color=BLUE, ax=axes[0])
axes[0].set_title('Delivered revenue by listing type')
axes[0].set_xlabel('revenue (millions)')
(top_loc / 1e6).plot(kind='barh', color=BLUE, ax=axes[1])
axes[1].set_title('Top 10 locations by delivered revenue')
axes[1].set_xlabel('revenue (millions)')
plt.tight_layout()
plt.savefig('images/05_revenue.png', dpi=200, bbox_inches='tight')
plt.show()

top_rest = (deliv.groupby(['RestaurantID', 'name', 'location'])
                 .agg(orders=('OrderID', 'count'), revenue=('OrderValue', 'sum'))
                 .nlargest(10, 'revenue'))
display(top_rest)

print('Share of revenue from the top 10% of restaurants:',
      f"{deliv.groupby('RestaurantID')['OrderValue'].sum().nlargest(int(deliv['RestaurantID'].nunique() * 0.1)).sum() / deliv['OrderValue'].sum():.1%}")


In [ ]:
kpi = {
    'Total orders':               f"{len(orders):,}",
    'Date range':                 f"{orders['OrderDate'].min().date()} to {orders['OrderDate'].max().date()}",
    'Active customers':           f"{orders['CustomerID'].nunique():,}",
    'Restaurants ordered from':   f"{orders['RestaurantID'].nunique():,}",
    'Delivered %':                f"{(orders['OrderStatus'] == 'Delivered').mean():.1%}",
    'Cancelled %':                f"{(orders['OrderStatus'] == 'Cancelled').mean():.1%}",
    'Delayed %':                  f"{(orders['OrderStatus'] == 'Delayed').mean():.1%}",
    'Average order value':        f"{orders['OrderValue'].mean():,.0f}",
    'Delivered revenue':          f"{orders.loc[orders['OrderStatus'] == 'Delivered', 'OrderValue'].sum():,.0f}",
    'Revenue lost to cancellations': f"{orders.loc[orders['OrderStatus'] == 'Cancelled', 'OrderValue'].sum():,.0f}",
    'Average delivery time (min)': f"{orders['DeliveryTimeMins'].mean():.1f}",
}
print(pd.Series(kpi).to_string())



In [ ]:
# Zip and download all saved charts
!zip -r images.zip images
from google.colab import files
files.download('images.zip')
